<a href="https://colab.research.google.com/github/WVF-1/Urban-Tree-Cover/blob/main/01_data_acquisition_merging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Setup

# Week 3 — What Influences Urban Tree Cover?
## Part 1 of 3: Building the Predictor Dataset

**Building Data Together Data Science Newsletter**

In Weeks 1–2 we profiled tree species diversity and native-species representation across 63 U.S. cities using the tree census reference table (Falasco et al., via Dryad). This week we ask a different question:

> **What city-level characteristics are associated with more (or less) urban tree cover?**

To answer that, we need to go get some variables that *aren't* in the original tree census: **elevation, population density, impervious surface, park land, and open water** — for each of the 63 cities.

This notebook does the data-engineering legwork:
1. Load the 63-city reference table
2. Geocode each city's administrative boundary (OpenStreetMap / Nominatim via `osmnx`)
3. Compute land area → population density
4. Pull point elevation (Open-Elevation / Open-Meteo APIs)
5. Compute % park land, % open water, and an OSM-derived impervious-surface *proxy* — all from OpenStreetMap land-use polygons clipped to each city boundary
6. Merge everything into one predictor table and export it (CSV + city boundary GeoJSON) for Notebooks 2 and 3

**No modeling happens in this notebook** — it's pure data acquisition. Every source is a free, no-API-key public dataset, so this is fully reproducible from a cold start.

⚠️ A note on honesty: some of these variables are *proxies*, not gold-standard measurements (see the caveats at the end). We're optimizing for "reproducible in a Colab notebook with no API keys," not "satellite-grade land cover classification." Treat this as exploratory infrastructure, not a peer-reviewed land-cover product.


In [1]:
# Colab-friendly setup. Safe to re-run.
# Explicitly uninstall and reinstall numpy and pandas to resolve potential binary incompatibility issues.
!pip uninstall -y numpy pandas
# Installing pandas==2.2.2 to satisfy google-colab's dependency, and let pip resolve numpy.
!pip install -q pandas==2.2.2 numpy osmnx==1.9.4 geopandas shapely pyproj requests tqdm

import time
import re
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import requests
from shapely.geometry import Point
from tqdm.auto import tqdm

ox.settings.use_cache = True
ox.settings.log_console = False
ox.settings.timeout = 180

print("osmnx", ox.__version__)
print("geopandas", gpd.__version__)

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
spopt 0.7.0 requires shapely>=2.1.0, but you have shapely 2.0.7 which is incompatible.
tobler 0.14.0 requires geopandas>=1.0, but you have geopandas 0.14.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but

## 1. Load the base reference table

In [2]:
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CSV_NAME = "city_reference_dryad_summary.csv"

if IN_COLAB:
    print(f"Upload {CSV_NAME} (from the Dryad tree census reference table)")
    uploaded = files.upload()
    CSV_NAME = list(uploaded.keys())[0]

base = pd.read_csv(CSV_NAME)
print(base.shape)
base.head()


Upload city_reference_dryad_summary.csv (from the Dryad tree census reference table)


Saving city_reference_dryad_summary.csv to city_reference_dryad_summary (1).csv
(63, 13)


,city,state,region,population,lat,long,tree_city_usa,ref_number_trees,ref_number_species,ref_effective_species,ref_percent_native,ref_most_common_species,ref_most_common_genus
0,Albuquerque,New Mexico,West,556495,35.085334,-106.605553,yes,2533,80,34.308079,0.254545,Ulmus pumila,Ulmus
1,Anaheim,California,West,345012,33.835293,-117.914504,yes,79651,291,52.411599,0.028760,Magnolia grandiflora,Magnolia
2,Arlington,Texas,South,379577,32.735687,-97.108066,yes,14827,104,31.429879,0.775998,Ulmus crassifolia,Quercus
3,Atlanta,Georgia,South,447841,33.748995,-84.387982,yes,41307,284,57.090891,0.742589,Cercis canadensis,Quercus
4,Aurora (CO),Colorado,West,345803,39.729432,-104.831919,yes,57658,119,21.535282,0.085738,Gleditsia triacanthos,Gleditsia


## 2. Geocode city boundaries

Some entries in the reference table use disambiguating suffixes like `Aurora (CO)` — these aren't valid Nominatim queries, so we strip the parenthetical and geocode on `city, state, USA` instead. We query one at a time with a short delay to stay well within Nominatim's usage policy (max ~1 request/second).


In [3]:
def clean_query(city, state):
    name = re.sub(r"\s*\([^)]*\)", "", city).strip()
    return f"{name}, {state}, USA"

boundaries = {}
failed = []

for _, row in tqdm(base.iterrows(), total=len(base), desc="Geocoding city boundaries"):
    q = clean_query(row["city"], row["state"])
    try:
        gdf = ox.geocode_to_gdf(q)
        # Nominatim can return multiple candidates for ambiguous names; keep the
        # first (highest-ranked) match, which is what geocode_to_gdf returns by default.
        boundaries[row["city"]] = gdf.iloc[0].geometry
    except Exception as e:
        failed.append((row["city"], str(e)))
    time.sleep(1.0)  # be a polite Nominatim citizen

print(f"Geocoded {len(boundaries)} / {len(base)} cities")
if failed:
    print("Failed:")
    for name, err in failed:
        print(f"  - {name}: {err}")


Geocoding city boundaries:   0%|          | 0/63 [00:00<?, ?it/s]

Geocoded 63 / 63 cities


In [4]:
# Build a GeoDataFrame of boundaries and reproject to an equal-area CRS
# (NAD83 / Conus Albers, EPSG:5070) to get honest area figures.
# Caveat: EPSG:5070 is defined for the conterminous US, so Honolulu's area
# figure is less reliable than the mainland cities' — flagged again below.

boundary_gdf = gpd.GeoDataFrame(
    {"city": list(boundaries.keys())},
    geometry=list(boundaries.values()),
    crs="EPSG:4326",
)

boundary_proj = boundary_gdf.to_crs(epsg=5070)
boundary_gdf["area_km2"] = boundary_proj.geometry.area / 1e6

boundary_gdf.head()


,city,geometry,area_km2
0,Albuquerque,"MULTIPOLYGON (((-106.69272 35.08326, -106.6928...",518.412144
1,Anaheim,"POLYGON ((-118.01736 33.81717, -118.01729 33.8...",131.681086
2,Arlington,"POLYGON ((-97.23382 32.68783, -97.23381 32.687...",258.253722
3,Atlanta,"MULTIPOLYGON (((-84.55085 33.72241, -84.55084 ...",352.256517
4,Aurora (CO),"MULTIPOLYGON (((-104.69826 39.68207, -104.6974...",417.152175


## 3. Elevation

In [5]:
def get_elevations_open_elevation(lats, lons, chunk_size=50):
    elevations = []
    for i in range(0, len(lats), chunk_size):
        locs = [{"latitude": la, "longitude": lo}
                for la, lo in zip(lats[i:i+chunk_size], lons[i:i+chunk_size])]
        try:
            resp = requests.post(
                "https://api.open-elevation.com/api/v1/lookup",
                json={"locations": locs},
                timeout=30,
            )
            resp.raise_for_status()
            results = resp.json()["results"]
            elevations.extend([r["elevation"] for r in results])
        except Exception as e:
            print("Open-Elevation batch failed:", e)
            elevations.extend([np.nan] * len(locs))
    return elevations


def get_elevations_open_meteo(lats, lons):
    lat_str = ",".join(str(x) for x in lats)
    lon_str = ",".join(str(x) for x in lons)
    url = f"https://api.open-meteo.com/v1/elevation?latitude={lat_str}&longitude={lon_str}"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return resp.json()["elevation"]


lats = base["lat"].tolist()
lons = base["long"].tolist()

try:
    elevations = get_elevations_open_elevation(lats, lons)
    if any(pd.isna(e) for e in elevations):
        raise ValueError("Open-Elevation returned partial NaNs, falling back")
except Exception as e:
    print("Falling back to Open-Meteo elevation API:", e)
    elevations = get_elevations_open_meteo(lats, lons)

base["elevation_m"] = elevations
base[["city", "elevation_m"]].head()


,city,elevation_m
0,Albuquerque,1587.0
1,Anaheim,61.0
2,Arlington,190.0
3,Atlanta,336.0
4,Aurora (CO),1652.0


## 4. Land cover shares: water, parks, and an impervious-surface proxy

For each city boundary we pull three OSM feature layers and compute what share of the city's land area each one covers:

- **`pct_water`** — `natural=water` + `waterway=riverbank` polygons
- **`pct_parkland`** — `leisure=park`, `leisure=nature_reserve`, `leisure=garden`
- **`pct_impervious_proxy`** — `landuse` in {residential, commercial, industrial, retail, construction}

That last one is explicitly a **proxy**, not the satellite-derived NLCD impervious-surface product — it's "how much of the city is OSM-tagged as built-up land use," which correlates with true imperviousness but isn't the same measurement. We're trading precision for a zero-API-key, zero-auth pipeline. If you want to swap in true NLCD imperviousness later, that requires Google Earth Engine (`ee.Authenticate()`) and city boundary zonal statistics — a good Week 4 upgrade, not a Week 3 blocker.


In [16]:
proj_epsg = 5070

WATER_TAGS = {"natural": "water", "waterway": "riverbank"}
PARK_TAGS = {"leisure": ["park", "nature_reserve", "garden"]}
IMPERVIOUS_TAGS = {"landuse": ["residential", "commercial", "industrial", "retail", "construction"]}

# Combine all tags into a single dictionary for a more efficient Overpass query
# osmnx.geometries.geometries_from_polygon interprets multiple top-level keys as an 'OR' query
ALL_TAGS = {}
ALL_TAGS.update(WATER_TAGS)
ALL_TAGS.update(PARK_TAGS)
ALL_TAGS.update(IMPERVIOUS_TAGS)

pct_water, pct_park, pct_impervious = [], [], []

for _, row in tqdm(boundary_gdf.iterrows(), total=len(boundary_gdf), desc="Land cover shares"):
    city_geom = row.geometry
    city_area_sqm = row["area_km2"] * 1e6 # Convert to square meters for area calculation

    try:
        # Fetch all relevant features for the city in a single API call
        # osmnx.geometries.geometries_from_polygon handles MultiPolygon geometries correctly
        all_features = ox.geometries.geometries_from_polygon(city_geom, tags=ALL_TAGS)

        if all_features.empty:
            pct_water.append(0.0)
            pct_park.append(0.0)
            pct_impervious.append(0.0)
            continue

        # Reproject all features to an equal-area CRS for accurate area calculations
        all_features_proj = all_features.to_crs(epsg=proj_epsg)
        city_proj = gpd.GeoSeries([city_geom], crs="EPSG:4326").to_crs(epsg=proj_epsg).iloc[0]

        # Clip features to the city boundary to ensure accurate area within the city
        # Note: clip can sometimes return an empty GeoDataFrame if no overlap or due to geometry issues
        clipped_features = gpd.clip(all_features_proj, city_proj)

        if clipped_features.empty:
            pct_water.append(0.0)
            pct_park.append(0.0)
            pct_impervious.append(0.0)
            continue

        # Calculate area for each category from the clipped features
        # Use .get() with default to handle cases where a tag column might be missing
        water_filter = (
            (clipped_features.get("natural") == "water") |
            (clipped_features.get("waterway") == "riverbank")
        )
        water_area = clipped_features[water_filter].geometry.area.sum() if not clipped_features[water_filter].empty else 0.0
        pct_water.append(water_area / city_area_sqm if city_area_sqm > 0 else 0.0)

        park_filter = clipped_features.get("leisure", pd.Series()).isin(PARK_TAGS["leisure"])
        park_area = clipped_features[park_filter].geometry.area.sum() if not clipped_features[park_filter].empty else 0.0
        pct_park.append(park_area / city_area_sqm if city_area_sqm > 0 else 0.0)

        impervious_filter = clipped_features.get("landuse", pd.Series()).isin(IMPERVIOUS_TAGS["landuse"])
        impervious_area = clipped_features[impervious_filter].geometry.area.sum() if not clipped_features[impervious_filter].empty else 0.0
        pct_impervious.append(impervious_area / city_area_sqm if city_area_sqm > 0 else 0.0)

    except Exception as e:
        print(f"Error processing city {row['city']}: {e}")
        pct_water.append(np.nan)
        pct_park.append(np.nan)
        pct_impervious.append(np.nan)

boundary_gdf["pct_water"] = pct_water
boundary_gdf["pct_parkland"] = pct_park
boundary_gdf["pct_impervious_proxy"] = pct_impervious

boundary_gdf[["city", "area_km2", "pct_water", "pct_parkland", "pct_impervious_proxy"]].head(10)

Land cover shares:   0%|          | 0/63 [00:00<?, ?it/s]

/tmp/ipykernel_6987/2774974267.py:23: FutureWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in the v2.0.0 release. See the OSMnx v2 migration guide: https://github.com/gboeing/osmnx/issues/1123
  all_features = ox.geometries.geometries_from_polygon(city_geom, tags=ALL_TAGS)
/usr/local/lib/python3.12/dist-packages/osmnx/_overpass.py:285: FutureWarning: `settings.timeout` is deprecated and will be removed in the v2.0.0 release: use `settings.requests_timeout` instead. See the OSMnx v2 migration guide: https://github.com/gboeing/osmnx/issues/1123
  overpass_settings = _make_overpass_settings()
/usr/local/lib/python3.12/dist-packages/osmnx/_overpass.py:395: FutureWarning: `settings.timeout` is deprecated and will be removed in the v2.0.0 release: use `settings.requests_timeout` instead. See the OSMnx v2 

Error processing city Austin: cannot access local variable 'response' where it is not associated with a value


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/osmnx/_overpass.py:285: FutureWarning: `settings.timeout` is deprecated and will be removed in the v2.0.0 release: use `settings.requests_timeout` instead. See the OSMnx v2 migration guide: https://github.com/gboeing/osmnx/issues/1123
  overpass_settings = _make_overpass_settings()
/usr/local/lib/python3.12/dist-packages/osmnx/_overpass.py:395: FutureWarning: `settings.timeout` is deprecated and will be removed in the v2.0.0 release: use `settings.requests_timeout` instead. See the OSMnx v2 migration guide: https://github.com/gboeing/osmnx/issues/1123
  yield _overpass_request(data={"data": query_str})
/usr/local/lib/python3.12/dis

Error processing city Dallas: cannot access local variable 'response' where it is not associated with a value


...

Error processing city Washington (DC): cannot access local variable 'response' where it is not associated with a value


/tmp/ipykernel_6987/2774974267.py:23: FutureWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in the v2.0.0 release. See the OSMnx v2 migration guide: https://github.com/gboeing/osmnx/issues/1123
  all_features = ox.geometries.geometries_from_polygon(city_geom, tags=ALL_TAGS)
/usr/local/lib/python3.12/dist-packages/osmnx/_overpass.py:285: FutureWarning: `settings.timeout` is deprecated and will be removed in the v2.0.0 release: use `settings.requests_timeout` instead. See the OSMnx v2 migration guide: https://github.com/gboeing/osmnx/issues/1123
  overpass_settings = _make_overpass_settings()
/usr/local/lib/python3.12/dist-packages/osmnx/_overpass.py:395: FutureWarning: `settings.timeout` is deprecated and will be removed in the v2.0.0 release: use `settings.requests_timeout` instead. See the OSMnx v2 

Error processing city Worcester: cannot access local variable 'response' where it is not associated with a value


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,city,area_km2,pct_water,pct_parkland,pct_impervious_proxy
0,Albuquerque,518.412144,0.008644,0.101283,0.428055
1,Anaheim,131.681086,0.028259,0.048618,0.433079
2,Arlington,258.253722,0.044521,0.035914,0.537761
3,Atlanta,352.256517,0.007924,0.037766,0.694056
4,Aurora (CO),417.152175,0.013026,0.036362,0.303586
5,Austin,724.844564,NaN,NaN,NaN
6,Baltimore,238.105637,0.122449,0.087712,0.464752
7,Boston,246.061160,0.020479,0.079160,0.137253
8,Buffalo,121.292631,0.140141,0.066706,0.153336
9,Cape Coral,306.571102,0.084210,0.017421,0.582765


## 5. Merge everything into one predictor table

In [17]:
merged = base.merge(
    boundary_gdf[["city", "area_km2", "pct_water", "pct_parkland", "pct_impervious_proxy"]],
    on="city", how="left",
)

merged["population_density_per_km2"] = merged["population"] / merged["area_km2"]

# Derived tree-cover-related outcome metrics for Notebook 2
merged["trees_per_km2"] = merged["ref_number_trees"] / merged["area_km2"]
merged["trees_per_1000_pop"] = merged["ref_number_trees"] / merged["population"] * 1000

n_missing = merged["area_km2"].isna().sum()
print(f"{n_missing} of {len(merged)} cities missing boundary/area data (geocoding failures above)")

merged.describe(include="all").T


0 of 63 cities missing boundary/area data (geocoding failures above)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
city,63,63,Albuquerque,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,63,33,California,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,63,4,West,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
population,63.0,NaN,NaN,NaN,668531.063492,1139943.50059,164676.0,240030.0,379577.0,640215.0,8405837.0
lat,63.0,NaN,NaN,NaN,36.983231,5.167133,21.306944,33.76145,37.774929,40.576704,47.606209
long,63.0,NaN,NaN,NaN,-97.712392,18.752018,-157.858333,-117.621998,-94.670792,-82.203355,-71.05888
tree_city_usa,63,2,yes,56,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ref_number_trees,63.0,NaN,NaN,NaN,89845.031746,134892.664132,214.0,15735.0,45148.0,98621.0,720140.0
ref_number_species,63.0,NaN,NaN,NaN,166.698413,103.586591,16.0,101.0,137.0,213.0,528.0
ref_effective_species,63.0,NaN,NaN,NaN,34.167181,17.011951,5.858471,21.683781,33.419524,45.262776,93.301271


In [18]:
OUT_CSV = "week3_city_predictors.csv"
merged.to_csv(OUT_CSV, index=False)

OUT_GEOJSON = "week3_city_boundaries.geojson"
boundary_gdf.to_file(OUT_GEOJSON, driver="GeoJSON")

print(f"Saved {OUT_CSV} and {OUT_GEOJSON}")

if IN_COLAB:
    files.download(OUT_CSV)
    files.download(OUT_GEOJSON)


Saved week3_city_predictors.csv and week3_city_boundaries.geojson


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Data quality notes (read before Notebook 2)

- **Elevation** is a single point reading at the city's reference lat/long, not an average across the city's full extent — cities with big elevation ranges (e.g. mountain-adjacent cities) will be under-characterized by a single number.
- **`pct_impervious_proxy`** is built from OSM `landuse` tagging density, which varies by how thoroughly each city has been mapped in OpenStreetMap. Well-mapped cities (large metros) will likely look more "complete" than smaller or less-mapped ones — this is a real source of noise, not signal.
- **Area / population density** comes from OSM administrative boundary polygons, which don't always match the U.S. Census Bureau's official place boundaries (used for the `population` figures in the base table) — there can be small city-limit mismatches.
- **`EPSG:5070`** (used for all area math) is defined for the conterminous U.S., so **Honolulu's** area and density figures are the least reliable in the table.
- Any city that failed to geocode will have `NaN`s across the new columns — Notebook 2 will need to decide whether to drop or impute them.

None of this disqualifies the data for *exploratory* analysis — it just means we shouldn't over-read small effect sizes or precise thresholds. That's consistent with this being Week 3 of an exploratory series, not a finished model.
